# Week 5 lecture walkthrough: choosing a filtered construction for the observed data type

This is the live worked example, built around a decision clinic comparing a state cloud, weighted interactions and a scalar field. It follows the conceptual argument of the slides through prediction, reveal and interpretation. It is not intended as a line-by-line answer key to the participant practical.

**Resource boundary.** The [reference notes](index.qmd) explain why data type precedes construction choice. The [slides](slides.qmd) present the decision map. This notebook demonstrates one complete workflow for each data type. The [participant practical](lab.ipynb) leaves one controlled modelling change and its interpretation to students.

**Lecture map.** Name the observed object before running software. For each of the point cloud, graph and scalar grid, identify the constructed complex, filtration direction and scientific meaning of the parameter.

This laboratory uses one software stack, GUDHI, for three input types: a point cloud, a weighted graph and a gridded scalar field. The purpose is not API coverage. It is to keep asking which observed object, complex and filtration answer the question.

All homology uses $\mathbb F_2$. See the **Applied glossary** for *simplex tree*, *cubical complex*, *lower-star filtration* and *clique complex*.

**Presenter route.** Treat the three synthetic inputs as one construction clinic. Keep asking which modelling claim turns each observation into a filtered complex.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
RNG=np.random.default_rng(3024)
import gudhi as gd

def finite_intervals(st,dim):
    D=st.persistence_intervals_in_dimension(dim)
    return D[np.isfinite(D[:,1])] if len(D) else D

def longest(D): return float(np.max(D[:,1]-D[:,0])) if len(D) else 0.0
print('GUDHI',gd.__version__)

## 1. Observe: lecture demonstration

**Presenter cue.** Show the object before the calculation. Ask the room to separate what is given from what will be constructed.

Three data objects are shown below: sampled states in a point cloud, weighted pairwise interactions in a graph, and a scalar field on a grid. Before computing, name what is directly observed and what must be constructed.

In [ ]:
theta=np.linspace(0,2*np.pi,70,endpoint=False)
cloud=np.c_[np.cos(theta),np.sin(theta)]+.055*RNG.normal(size=(70,2))

graph_edges={(0,1):.3,(1,2):.4,(0,2):.45,(2,3):.7,(3,4):.8,(2,4):.85}

x=np.linspace(-2,2,45); X,Y=np.meshgrid(x,x)
field=(X**2+Y**2-1.0)**2

fig,axes=plt.subplots(1,3,figsize=(12,3.3))
axes[0].scatter(*cloud.T,s=12); axes[0].set_aspect('equal'); axes[0].set_title('sampled state cloud')
graph_pos={0:(0,1),1:(1,1.4),2:(2,1),3:(2.8,.4),4:(3.7,.7)}
for (i,j),weight in graph_edges.items():
    axes[1].plot([graph_pos[i][0],graph_pos[j][0]],[graph_pos[i][1],graph_pos[j][1]],color='0.35')
    midpoint=((graph_pos[i][0]+graph_pos[j][0])/2,(graph_pos[i][1]+graph_pos[j][1])/2)
    axes[1].text(*midpoint,f'{weight:.2f}',fontsize=8,color='tab:blue')
for vertex,xy in graph_pos.items():
    axes[1].scatter(*xy,s=90,zorder=3); axes[1].text(xy[0],xy[1]+.13,str(vertex),ha='center')
axes[1].set_title('weighted interaction graph'); axes[1].axis('off')
axes[2].imshow(field,origin='lower',extent=[-2,2, -2,2]); axes[2].set_title('scalar field')
plt.show()

## 2. Predict: lecture demonstration

**Presenter cue.** Pause here and collect at least two predictions before revealing any output.

1. Which $H_1$ feature should the point cloud contain?
2. In the graph, should a three-clique remain a graph cycle or be filled as a 2-simplex?
3. For the field, what do low sublevel values represent geometrically?
4. Which maximum homology dimension and filtration cutoff are actually needed?

## 3. Implement: lecture demonstration

**Reveal.** Run one cell at a time. Name the domain, codomain, complex, module or summary before interpreting its values.

### A. Point cloud: Rips filtration

GUDHI uses the pairwise-distance threshold as the Rips filtration value. Construct only through dimension 2 because that is enough to calculate $H_1$.

In [ ]:
rips=gd.RipsComplex(points=cloud,max_edge_length=1.8)
st_point=rips.create_simplex_tree(max_dimension=2)
st_point.compute_persistence(homology_coeff_field=2)
D1_point=finite_intervals(st_point,1)
print('simplices:',st_point.num_simplices(),'longest finite H1:',round(longest(D1_point),3))

### B. Weighted graph: graph or clique complex?

Keep the vertices and edge weights fixed. First retain only the graph. Then expand every clique to dimension 2. This changes the mathematical object, not the observations.

In [ ]:
def weighted_graph_tree(fill_cliques):
    st=gd.SimplexTree()
    for v in range(5): st.insert([v],filtration=0.0)
    for e,w in graph_edges.items(): st.insert(e,filtration=w)
    if fill_cliques: st.expansion(2)
    st.make_filtration_non_decreasing(); st.compute_persistence(homology_coeff_field=2,persistence_dim_max=True)
    return st

graph_st=weighted_graph_tree(False)
clique_st=weighted_graph_tree(True)
for name,st in [('graph',graph_st),('clique complex',clique_st)]:
 print(name,'simplices',st.num_simplices(),'H1',st.persistence_intervals_in_dimension(1))

### C. Scalar field: cubical sublevel filtration

The top-dimensional cells carry field values. In two dimensions, each array entry is a filled square 2-cell whose boundary edges and corner vertices are included with it. Low values enter first, so persistence coordinates are field values rather than distances. Negating the field would change the question to a superlevel analysis.

In [ ]:
cubical=gd.CubicalComplex(top_dimensional_cells=field)
cubical.compute_persistence(homology_coeff_field=2)
print('cubical dimension',cubical.dimension())
print('longest finite H1',round(longest(cubical.persistence_intervals_in_dimension(1)),3))

## 4. Compare: lecture demonstration

**Controlled comparison.** Keep the stated input fixed and change only the highlighted modelling decision.

Change one upstream decision at a time:

- point cloud: halve the sample size;
- graph: switch clique filling on;
- field: negate the values.

Predict first. Which changes test numerical sensitivity, and which redefine the scientific question?

In [ ]:
small=cloud[::2]
small_st=gd.RipsComplex(points=small,max_edge_length=1.8).create_simplex_tree(max_dimension=2)
small_st.compute_persistence(homology_coeff_field=2)
print('full/subsampled longest H1',round(longest(D1_point),3),round(longest(finite_intervals(small_st,1)),3))
for name,F in [('sublevel',field),('superlevel via -field',-field)]:
    cc=gd.CubicalComplex(top_dimensional_cells=F); cc.compute_persistence(homology_coeff_field=2)
    print(name,'longest H1',round(longest(cc.persistence_intervals_in_dimension(1)),3))

## 5. Interpret: lecture demonstration

**Presenter close.** Ask what the example supports, what information was discarded and which stronger claim would be unjustified.

1. Which output belongs to a point-cloud representation, which to an interaction model, and which to a field?
2. When is clique filling scientifically defensible?
3. What is the physical meaning of a cubical birth value?
4. Why does computing $H_1$ require the complex through dimension 2, and why would $H_2$ require tetrahedra?
5. Which resource limits and constructed simplex or cell counts should be reported?
6. What simpler baseline belongs beside each topology calculation?

**† Qualification.** Every example is synthetic. The workflows demonstrate consequences of choices, not empirical performance.

The point-cloud result concerns sampled metric organisation. Graph and clique results differ because clique filling asserts higher-order coherence. Field and negated-field results organise around lows and highs respectively. Subsampling is a sensitivity check; changing filtration direction or filling rule changes the question.

## Lecture close

Return to the final slide questions.

1. Name the observed or starting object.
2. Name every constructed object used in this walkthrough.
3. Identify the single modelling decision that drove the central comparison.
4. State one conclusion supported by the calculation and one conclusion it cannot establish.

**Take-forward example.** The purpose of a decision clinic comparing a state cloud, weighted interactions and a scalar field is to make choosing a filtered construction for the observed data type concrete. The example is deliberately small or synthetic so that the construction remains inspectable.